### 08. Z-score에 넣을 센서값과 조회 정보를 나누기

- 선택한 CSV에서 아래 `INPUT_COLUMNS`의 센서값을 읽음.
  - 점수 계산과 모델 학습은 하지 않음.
- `X_zscore`에는 센서값만, `row_context`에는 LOT, 시각, 세 라벨을 둠.
  - 두 표는 같은 행 순서로 저장함.
- 결측을 채우거나 전체 행을 버리지 않음.
  - `valid_sensor`에 센서별 값이 있는지 따로 남김.


### CSV와 라벨 선택

- `DATA_FILE`의 파일명만 바꾸면 v1~v5를 읽음.
  - 기본 `LABEL_GROUP="Z4"`는 같은 구역끼리 비교하기 위한 선택임.
  - `CLN`, `Z1`~`Z9`, `COOLING`으로 바꿀 수 있음.
  - 예전 v1/v2의 행 전체 실험은 `ROW`로 선택함. v3~v5에는 행 전체 라벨이 없음.
- `STAT_LABEL`은 공통 움직임 해제 전 통계 결과.
  - `DOMAIN_LABEL`은 검토한 판단. 빈칸을 정상 `0`으로 채우지 않음.
  - 이 노트북은 해당 구역의 기존 `LABEL`을 분석에 사용함.
- 세 라벨과 근거 컬럼은 모델 입력 `X`에 넣지 않음.
  - v1~v3는 원래 기록, v4/v5는 1분 집계이므로 행수가 다름.


In [ ]:
DATA_FILE = "cal01_base_v5.csv"  # v1~v5 중 읽을 파일명
LABEL_GROUP = "Z4"  # CLN, Z1~Z9, COOLING. v1/v2의 예전 행 전체 실험은 ROW
LABEL_COL = LABEL_GROUP + "__LABEL"
if LABEL_GROUP == "ROW":
    LABEL_COL = "LABEL"

import os
import json
import hashlib
import pandas as pd
from IPython.display import display

# 저장소 루트, notebooks, namelesspark 폴더에서 같은 data 폴더를 읽음.
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, 'data')

ENC = 'utf-8-sig'
BASE_VERSION = os.path.splitext(DATA_FILE)[0]
INPUT_VERSION = 'v1'

BASE_NAME = DATA_FILE

BASE_PATH = os.path.join(DATA_DIR, 'processed', BASE_NAME)
OUT = os.path.join(DATA_DIR, 'processed', 'classification', 'zscore', os.path.splitext(DATA_FILE)[0] + "__" + LABEL_GROUP)
X_CSV = os.path.join(OUT, 'X.csv')
CONTEXT_CSV = os.path.join(OUT, 'row_context.csv')

INPUT_COLUMNS = ['TC-Z1', 'TC-Z2', 'TC-Z4', 'TC-Z5', 'TC-Z6', 'OP-Z1', 'OP-Z2', 'CP-Z4']
LABEL_PREFIX = "" if LABEL_GROUP == "ROW" else LABEL_GROUP + "__"
CONTEXT_COLUMNS = ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE"]
for suffix in ["STAT_LABEL", "DOMAIN_LABEL", "LABEL", "LABEL_STATUS"]:
    CONTEXT_COLUMNS.append(LABEL_PREFIX + suffix)


## 넣고 빼는 기준

아래 범위는 사용자 제외 구간을 뺀 관측 요약이며 정상 허용치가 아님. 현재 0도 정상 확정이 아님.

| 컬럼 | 이번 X | 이유 |
| --- | --- | --- |
| TC-Z1, Z2 | 후보로 포함 | LOT 중앙값이 약 100으로 비슷함. 강종, 운전단계별 변동 폭은 별도 확인 |
| TC-Z4, Z5, Z6 | 후보로 포함 | LOT 중앙값이 약 860 부근. 강종별 기준으로 수준 이탈 비교 |
| OP-Z1, Z2 | 보조 후보로 포함 | 실제 관측에 0, 100 포화 없음. 대응 TC와 함께 해석하며 출력만으로 고장 판정 안 함 |
| CP-Z4 | 후보로 포함 | LOT 중앙값은 약 0.45지만 분포 폭은 다름. 꼬리, 운전 전환을 정상 기준으로 섞지 않기 |
| CLN-IN | 보류 | 같은 SPHC에서도 LOT 중앙값 차이가 약 4.87. 강종만으로 설명 부족 |
| TC-Z3, OP-Z3 | 보류 | 제어 진동 맥락 확인 필요. OP-Z3의 0, 100, TC-Z3의 ±20을 곧바로 이상 또는 정상으로 확정하지 않음 |
| OP-Z4, Z5, Z6, Z7, Z8 | 보류 | 같은 강종에서도 출력 수준이 다름. 전부 포화형이어서 빼는 것은 아님 |
| TC-Z7, Z8 | 보류 | SPHC 240091의 수준이 다른 LOT보다 약 3 높음. 설정값 차이인지 미확인 |
| TC-Z9, TC-OUT1, OUT2 | 보류 | 느린 변화, 반복 패턴과 관측 위상 확인 필요. TC-Z9 전 LOT의 주기가 확인된 것은 아님 |
| CP-Z4M | 제외 | 의미 미확인 보조값. TC-Z5와 중복 가능성도 있음 |
| 1분전, 5분전, 직전10분평균 | 이번에는 제외 | 원값 비교와 분리. 센서마다 중복 정도가 달라 일괄 폐기하지 않음 |
| 직전두분차이, 직전10분표준편차 | 별도 검토 | 1분 평균으로 만든 변화, 변동 후보. 실제 10초 지속 여부는 복원 못 함 |
| OP-Z3 직전10분포화율 | 보류 | 제어 맥락용. 포화 자체를 고장으로 해석 못 함 |
| OP-Z4, Z5 직전10분포화율 | 제외 | 현재 유효값 전부 0 |
| GRADE, GRADE_*, LOT_ELAPSED_MIN | 점수 입력에서 제외 | 강종은 기준 구분용, 경과시간은 운전 맥락용 |
| LABEL, RULE, EVIDENCE_*, 판정/문맥 플래그 | 점수 입력에서 제외 | 평가, 근거 확인용. 숫자나 원핫으로 바꿔도 입력에 넣지 않음 |
| ROW_ID, LOT_NO, MEAS_DT, FEATURE_MINUTE | 점수 입력에서 제외 | 원본 연결, 시간순서, 분할용 |

분포 근거: 04, 05의 `analysis/grade/v1/`, `analysis/signals/v1/`. 파생 정의는 06에서 관리함.

In [ ]:
# 사용한 기준 파일의 해시를 저장 정보에 남김.
base_digest = hashlib.sha256()

with open(BASE_PATH, 'rb') as file:
    block = file.read(1024 * 1024)

    while block:
        base_digest.update(block)
        block = file.read(1024 * 1024)

base_hash = base_digest.hexdigest()

read_columns = CONTEXT_COLUMNS + ['EXCLUDED'] + INPUT_COLUMNS

master = pd.read_csv(
    BASE_PATH, encoding=ENC, usecols=read_columns, parse_dates=['MEAS_DT'],
    dtype={'ROW_ID': 'string', 'LOT_NO': 'int64', 'GRADE': 'string',
           'EXCLUDED': 'boolean'},
)

# 사용자 제외 구간만 뺌. 기존 RF 후보 67개 기준으로 행을 버리지 않음.
retained = master.loc[master['EXCLUDED'] == False].copy()

retained = retained.sort_values(['LOT_NO', 'MEAS_DT']).reset_index(drop=True)


## 수치 입력과 비교 정보 분리

결측을 0으로 채우거나 행 전체를 버리지 않음. 센서별 유효 여부는 `valid_sensor`로 확인하고, 결측 센서의 점수는 추후 계산 대상에서 빼야 함.
두 CSV는 **같은 행 순서로 한 쌍**임. 따로 정렬하지 않음. 라벨과 근거는 원본 기준표에서 ROW_ID로 다시 확인 가능함.

In [ ]:
X_zscore = retained[INPUT_COLUMNS].apply(pd.to_numeric, errors='raise').astype('float64')

valid_sensor = X_zscore.notna()

row_context = retained[CONTEXT_COLUMNS].copy()

row_context['ZSCORE_VALID_SENSOR_COUNT'] = valid_sensor.sum(axis=1).astype('int8')

row_context['ZSCORE_ALL_INPUTS_PRESENT'] = valid_sensor.all(axis=1)

grade_review = row_context.groupby('GRADE', sort=True).agg(
    LOT_COUNT=('LOT_NO', 'nunique'), ROWS=('ROW_ID', 'size'),
    ALL_INPUTS_PRESENT=('ZSCORE_ALL_INPUTS_PRESENT', 'sum'),
)

grade_review['CROSS_LOT_COMPARISON_POSSIBLE'] = grade_review['LOT_COUNT'] >= 2

# LOT가 여러 개여도 아직 학습/시험을 나누지 않았으므로 학습 기준 확보를 뜻하지 않음.
display(grade_review)
display(pd.DataFrame({'유효 행': valid_sensor.sum(), '결측 행': X_zscore.isna().sum()}))
display(X_zscore.head())


## 강종별 기준은 다음 단계에서

- 강종별 LOT 수: SPHC 4, SPCE 3, SPCEN, SPFH590 2, SGCC, DP590, SPCD 1. 단일 LOT 강종은 다른 LOT로 검증 불가.
- 시간, LOT 분할 후 **학습 구간에서만** 센서별 강종별 기준 평균, 표준편차를 정함. 현재 LABEL 0 전부를 정상 기준으로 확정하지 않음.
- 같은 강종이라도 설정값, 운전단계가 다를 수 있음. 설정값 미확인 상태이며, 시험 LOT 자체 평균으로 맞춰 지속 수준 차이를 지우지 않음.
- 기준은 9:1로 표본을 줄이기 전에 검토함. 학습에 없는 강종, 0, 극소 표준편차, 문턱, 지속시간, 센서 종합 판정은 아직 미정.
- 08은 입력 준비까지만 함. 현재 임시 LABEL과 비교해도 실제 고장 탐지 성능을 확정할 수는 없음.

In [ ]:
missing_counts = {}

for column in INPUT_COLUMNS:
    missing_counts[column] = int(X_zscore[column].isna().sum())

label_counts = {}

for label, count in row_context[LABEL_COL].value_counts().sort_index().items():
    label_counts[str(int(label))] = int(count)

contract = {
    'input_version': INPUT_VERSION,
    'base_version': BASE_VERSION,
    'master_relative_path': '../../../' + BASE_NAME,
    'master_sha256': base_hash,
    'input_status': 'first_level_comparison_candidates_not_validated',
    'input_columns': INPUT_COLUMNS,
    'context_columns': row_context.columns.tolist(),
    'row_filter': 'EXCLUDED == False; do not filter by RF MODEL_ELIGIBLE',
    'row_order': ['LOT_NO', 'MEAS_DT'],
    'alignment': 'X.csv row i corresponds to row_context.csv row i; keep the pair together',
    'rows': len(X_zscore),
    'all_inputs_present_rows': int(valid_sensor.all(axis=1).sum()),
    'missing_by_sensor': missing_counts,
    'label_counts': label_counts,
    'label_kind': LABEL_COL,
    'label_used_for_input_or_row_selection': False,
    'missing_policy': 'preserve NaN; missing sensor has no score; no zero fill or interpolation',
    'group_context': 'GRADE; setting and operating phase remain unconfirmed',
    'reference_policy': 'train only, before downsampling; LABEL 0 is not a confirmed normal reference',
    'split_status': 'not_assigned',
    'sampling_applied': False,
    'reference_statistics_fitted': False,
    'scores_calculated': False,
    'labels_changed': False,
    'models_trained': False,
    'coverage_limit': '8 raw measurement candidates only; does not cover faults in all 21 sensors',
    'selection_notebook': 'notebooks/namelesspark/08_zscore_dataframe.ipynb',
}

os.makedirs(OUT, exist_ok=True)

manifest_path = os.path.join(OUT, 'dataset.json')

X_zscore.to_csv(X_CSV, index=False, encoding=ENC)

row_context.to_csv(CONTEXT_CSV, index=False, encoding=ENC)

contract['file_sha256'] = {}

for name in ['X.csv', 'row_context.csv']:
    output_path = os.path.join(OUT, name)

    output_digest = hashlib.sha256()

    with open(output_path, 'rb') as file:
        block = file.read(1024 * 1024)

        while block:
            output_digest.update(block)
            block = file.read(1024 * 1024)

    contract['file_sha256'][name] = output_digest.hexdigest()

with open(manifest_path, 'w', encoding='utf-8') as file:
    json.dump(contract, file, ensure_ascii=False, indent=2)

print('저장:', os.path.relpath(OUT, ROOT))
print('X_zscore:', X_zscore.shape, '/ 임시 LABEL:', contract['label_counts'])
print('입력 준비만 완료. 기준 통계, 점수, 분할, 샘플링, 모델은 실행하지 않았음.')


## 다시 읽을 때

X와 비교 정보를 같은 행 순서로 같이 읽음. 강종은 `context_saved.GRADE`, 평가용 임시 라벨은 `context_saved.LABEL`임. 비교 정보 전체를 Z-score에 넣지 않음.


In [ ]:
X_saved = pd.read_csv(X_CSV, encoding=ENC)

context_saved = pd.read_csv(
    CONTEXT_CSV, encoding=ENC, parse_dates=['MEAS_DT'],
    dtype={'ROW_ID': 'string', 'LOT_NO': 'int64', 'GRADE': 'string',
           'LABEL': 'Int8', 'LABEL_STATUS': 'string', 'ZSCORE_VALID_SENSOR_COUNT': 'int8'},
)
